# Chinese Understanding Benchmark v10

Corrected version based on the working v6 behavior.

Changes:
- Restores v6-style prompt/parser behavior.
- Loads each model once, evaluates all tasks, then clears memory.
- Uses fixed manual mapping for AFQMC and CMNLI.
- Uses dataset-derived mapping for TNEWS.
- Adds `mlx-community/gemma-4-e4b-it-4bit` first using MLX.


## 1. Install dependencies

In [1]:
!pip uninstall -y torchaudio torchvision mlx-vlm || true
!pip install -U transformers datasets accelerate peft trl scikit-learn pandas tqdm sentencepiece
!pip install -U mlx==0.31.1 mlx-lm==0.31.2 mlx-metal==0.31.1


## 2. Configuration

In [2]:
MODELS = {
    "gemma_e4b_it_4bit": {
        "model_id": "mlx-community/gemma-4-e4b-it-4bit",
        "backend": "mlx",
    },
    "qwen3_4b_instruct_2507": {
        "model_id": "Qwen/Qwen3-4B-Instruct-2507",
        "backend": "transformers",
    },
    "gemma_e2b_it": {
        "model_id": "google/gemma-4-E2B-it",
        "backend": "transformers",
    },
}

TASKS = ["afqmc", "tnews", "cmnli"]
SPLIT = "validation"
MAX_SAMPLES = 200
DEBUG_N = 3


## 3. Imports and cleanup helpers

In [3]:
import gc
import os
import re
import time
import warnings

import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")


def cleanup_memory():
    gc.collect()
    try:
        torch.mps.empty_cache()
    except Exception:
        pass
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass


def clear_model(model=None, tokenizer=None):
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass
    cleanup_memory()


## 4. Label mappings

In [4]:
FIXED_TASK_SPECS = {
    "afqmc": {
        "label_id_to_name": {"0": "不同", "1": "相同"},
        "label_name_to_id": {"不同": "0", "相同": "1"},
        "aliases": {
            "不同": ["不同", "不相同", "不一致", "不等价", "语义不同", "否", "no", "false", "0"],
            "相同": ["相同", "一致", "等价", "同义", "语义相同", "是", "yes", "true", "1"],
        },
        "label_id_style": "fixed_manual",
    },
    "cmnli": {
        # Your observed CLUE/HF mapping:
        # 0 = neutral, 1 = entailment, 2 = contradiction
        "label_id_to_name": {"0": "中立", "1": "蕴含", "2": "矛盾"},
        "label_name_to_id": {"中立": "0", "蕴含": "1", "矛盾": "2"},
        "aliases": {
            "中立": ["中立", "无法判断", "不能确定", "无关", "neutral", "0"],
            "蕴含": ["蕴含", "包含", "推出", "可以推出", "entailment", "entails", "1"],
            "矛盾": ["矛盾", "冲突", "contradiction", "contradict", "2"],
        },
        "label_id_style": "fixed_manual",
    },
}


# def make_tnews_spec(dataset):
#     names = dataset.features["label"].names
#     return {
#         "label_id_to_name": {str(i): name for i, name in enumerate(names)},
#         "label_name_to_id": {name: str(i) for i, name in enumerate(names)},
#         "aliases": {name: [name] for name in names},
#         "label_id_style": "dataset_features_tnews",
#     }
def make_tnews_spec(dataset):
    # Correct CLUE TNEWS mapping for Hugging Face `clue/tnews`
    names = [
        "故事",  # 0
        "文化",  # 1
        "娱乐",  # 2
        "体育",  # 3
        "财经",  # 4
        "房产",  # 5
        "汽车",  # 6
        "教育",  # 7
        "科技",  # 8
        "国际",  # 9
        "股票",  # 10
        "旅游",  # 11
        "军事",  # 12
        "农业",  # 13
        "电竞",  # 14
    ]

    return {
        "label_id_to_name": {str(i): name for i, name in enumerate(names)},
        "label_name_to_id": {name: str(i) for i, name in enumerate(names)},
        "aliases": {name: [name] for name in names},
        "label_id_style": "fixed_manual_tnews",
    }


def get_task_spec(task, dataset):
    if task == "tnews":
        return make_tnews_spec(dataset)
    return FIXED_TASK_SPECS[task]


def label_id_to_name(label_id, spec):
    return spec["label_id_to_name"].get(str(label_id), str(label_id))


def pred_id_to_name(pred_id, spec):
    if pred_id == "__invalid__":
        return "__invalid__"
    return spec["label_id_to_name"].get(str(pred_id), "__invalid__")


## 5. Prompts and parser

In [5]:
def build_prompt(task, ex, spec):
    labels = "、".join(spec["label_name_to_id"].keys())

    if task == "afqmc":
        return f"""你是中文二分类器。只输出“相同”或“不同”其中一个词，不要解释。

句子1：{ex["sentence1"]}
句子2：{ex["sentence2"]}

语义是否相同？答案："""

    if task == "cmnli":
        return f"""你是中文自然语言推理分类器。只输出“蕴含”、“中立”或“矛盾”其中一个词，不要解释。

前提：{ex["sentence1"]}
假设：{ex["sentence2"]}

关系是？答案："""

    if task == "tnews":
        return f"""你是中文新闻标题分类器。只能从以下类别中选择一个输出，不要解释。

可选类别：{labels}

标题：{ex["sentence"]}

类别："""

    raise ValueError(f"Unsupported task: {task}")


def normalize_output(text):
    text = str(text).strip().lower()

    # Strip common chat/template artifacts from MLX or Transformers outputs.
    for token in ["<bos>", "<eos>", "<pad>", "<start_of_turn>", "<end_of_turn>", "model", "assistant", "user"]:
        text = text.replace(token, " ")

    text = text.replace("答案：", " ").replace("答案:", " ")
    text = text.replace("类别：", " ").replace("类别:", " ")
    text = text.replace("标签：", " ").replace("标签:", " ")
    text = text.replace("\n", " ")
    text = re.sub(r"[。，“”，、；;:：\[\]\(\)（）\"'`]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_pred_id(task, raw_output, spec):
    text_norm = normalize_output(raw_output)

    # Exact normalized alias match first.
    for canonical_name, aliases in spec["aliases"].items():
        for alias in aliases:
            if normalize_output(alias) == text_norm:
                return spec["label_name_to_id"].get(canonical_name, "__invalid__")

    # Substring match second.
    # TNEWS aliases are only official category names, so "游戏" remains invalid instead of being silently remapped to "电竞".
    alias_items = []
    for canonical_name, aliases in spec["aliases"].items():
        for alias in aliases:
            alias_items.append((canonical_name, normalize_output(alias)))

    alias_items = sorted(alias_items, key=lambda x: len(x[1]), reverse=True)

    for canonical_name, alias_norm in alias_items:
        if alias_norm and alias_norm in text_norm:
            return spec["label_name_to_id"].get(canonical_name, "__invalid__")

    return "__invalid__"


## 6. Model backends

In [6]:
def load_transformers_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

    model.eval()
    return tokenizer, model


@torch.no_grad()
def generate_transformers(tokenizer, model, prompt, max_new_tokens=8):
    inputs = tokenizer(prompt, return_tensors="pt")

    try:
        inputs = inputs.to(model.device)
    except Exception:
        pass

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    return tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()


def load_mlx_model(model_id):
    from mlx_lm import load as mlx_load
    model, tokenizer = mlx_load(model_id)
    return tokenizer, model


def generate_mlx(tokenizer, model, prompt, max_new_tokens=8):
    from mlx_lm import generate as mlx_generate

    raw = mlx_generate(
        model,
        tokenizer,
        prompt=prompt,
        max_tokens=max_new_tokens,
        verbose=False,
    )

    return str(raw).strip()


## 7. Evaluation

In [7]:
def evaluate_loaded_model(model_key, model_id, backend, tokenizer, model):
    model_summaries = []
    model_rows = []

    for task in TASKS:
        dataset = load_dataset("clue", task, split=SPLIT)
        dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
        spec = get_task_spec(task, dataset)

        y_true = []
        y_pred = []

        start = time.time()

        for idx, ex in enumerate(tqdm(dataset, desc=f"{model_key}/{task}")):
            prompt = build_prompt(task, ex, spec)

            if backend == "transformers":
                raw = generate_transformers(tokenizer, model, prompt, max_new_tokens=8)
            elif backend == "mlx":
                raw = generate_mlx(tokenizer, model, prompt, max_new_tokens=8)
            else:
                raise ValueError(f"Unsupported backend: {backend}")

            gold_id = str(ex["label"])
            gold_name = label_id_to_name(gold_id, spec)

            pred_id = extract_pred_id(task, raw, spec)
            pred_name = pred_id_to_name(pred_id, spec)

            y_true.append(gold_id)
            y_pred.append(pred_id)

            row = {
                "model_key": model_key,
                "model_id": model_id,
                "backend": backend,
                "task": task,
                "split": SPLIT,
                "idx": idx,
                "gold_id": gold_id,
                "gold_name": gold_name,
                "raw": repr(raw),
                "raw_normalized": normalize_output(raw),
                "pred_name": pred_name,
                "pred_id": pred_id,
            }

            model_rows.append(row)

            if idx < DEBUG_N:
                print(row)

        elapsed = time.time() - start
        invalid_count = sum(p == "__invalid__" for p in y_pred)
        y_pred_for_score = [p if p != "__invalid__" else "-1" for p in y_pred]

        summary = {
            "model_key": model_key,
            "model_id": model_id,
            "backend": backend,
            "task": task,
            "split": SPLIT,
            "samples": len(y_true),
            "accuracy": accuracy_score(y_true, y_pred_for_score),
            "macro_f1": f1_score(y_true, y_pred_for_score, average="macro", zero_division=0),
            "invalid_rate": invalid_count / len(y_pred),
            "invalid_count": invalid_count,
            "seconds": elapsed,
            "samples_per_second": len(y_true) / elapsed if elapsed > 0 else None,
            "label_id_style": spec["label_id_style"],
        }

        print(summary)
        model_summaries.append(summary)

    return model_summaries, model_rows


def evaluate_one_model(model_key, model_cfg):
    model_id = model_cfg["model_id"]
    backend = model_cfg["backend"]

    cleanup_memory()
    print(f"\n=== Loading {model_key}: {model_id} [{backend}] ===")

    tokenizer = None
    model = None

    try:
        if backend == "transformers":
            tokenizer, model = load_transformers_model(model_id)
        elif backend == "mlx":
            tokenizer, model = load_mlx_model(model_id)
        else:
            raise ValueError(f"Unsupported backend: {backend}")

        summaries, rows = evaluate_loaded_model(
            model_key=model_key,
            model_id=model_id,
            backend=backend,
            tokenizer=tokenizer,
            model=model,
        )

    finally:
        print(f"Clearing model from memory: {model_key}")
        clear_model(model, tokenizer)

    return summaries, rows


## 8. Run benchmark

In [8]:
all_summaries = []
all_rows = []

for model_key, model_cfg in MODELS.items():
    summaries, rows = evaluate_one_model(model_key, model_cfg)
    all_summaries.extend(summaries)
    all_rows.extend(rows)

summary_df = pd.DataFrame(all_summaries)
detail_df = pd.DataFrame(all_rows)

summary_df



=== Loading gemma_e4b_it_4bit: mlx-community/gemma-4-e4b-it-4bit [mlx] ===


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

gemma_e4b_it_4bit/afqmc:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'gemma_e4b_it_4bit', 'model_id': 'mlx-community/gemma-4-e4b-it-4bit', 'backend': 'mlx', 'task': 'afqmc', 'split': 'validation', 'idx': 0, 'gold_id': '0', 'gold_name': '不同', 'raw': "'双十一花呗提额在哪'", 'raw_normalized': '双十一花呗提额在哪', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e4b_it_4bit', 'model_id': 'mlx-community/gemma-4-e4b-it-4bit', 'backend': 'mlx', 'task': 'afqmc', 'split': 'validation', 'idx': 1, 'gold_id': '0', 'gold_name': '不同', 'raw': "'花呗付款\\n\\n句子1：花'", 'raw_normalized': '花呗付款 句子1 花', 'pred_name': '相同', 'pred_id': '1'}
{'model_key': 'gemma_e4b_it_4bit', 'model_id': 'mlx-community/gemma-4-e4b-it-4bit', 'backend': 'mlx', 'task': 'afqmc', 'split': 'validation', 'idx': 2, 'gold_id': '1', 'gold_name': '相同', 'raw': "'我到支付宝实体店消费用花'", 'raw_normalized': '我到支付宝实体店消费用花', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e4b_it_4bit', 'model_id': 'mlx-community/gemma-4-e4b-it-4bit', 'backend': 'mlx', 'task': 'afqmc', 'spli

gemma_e4b_it_4bit/tnews:   0%|          | 0/200 [00:00<?, ?it/s]

...

gemma_e2b_it/tnews:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'tnews', 'split': 'validation', 'idx': 0, 'gold_id': '2', 'gold_name': '娱乐', 'raw': "'故事、文化、娱乐、体育、'", 'raw_normalized': '故事 文化 娱乐 体育', 'pred_name': '故事', 'pred_id': '0'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'tnews', 'split': 'validation', 'idx': 1, 'gold_id': '9', 'gold_name': '国际', 'raw': "'故事、文化、娱乐、体育、'", 'raw_normalized': '故事 文化 娱乐 体育', 'pred_name': '故事', 'pred_id': '0'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'tnews', 'split': 'validation', 'idx': 2, 'gold_id': '4', 'gold_name': '财经', 'raw': "'故事：：：：：：：'", 'raw_normalized': '故事', 'pred_name': '故事', 'pred_id': '0'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'tnews', 'split': 'validation', 'samples': 200, 'accuracy': 0.015, 'macro_f1': 0.0018656716417

gemma_e2b_it/cmnli:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'cmnli', 'split': 'validation', 'idx': 0, 'gold_id': '0', 'gold_name': '中立', 'raw': "'每个人都很喜欢最新的福利\\n\\n答案：'", 'raw_normalized': '每个人都很喜欢最新的福利', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'cmnli', 'split': 'validation', 'idx': 1, 'gold_id': '1', 'gold_name': '蕴含', 'raw': "'嗯，我不知道，我不知道，'", 'raw_normalized': '嗯 我不知道 我不知道', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'cmnli', 'split': 'validation', 'idx': 2, 'gold_id': '2', 'gold_name': '矛盾', 'raw': "'我最喜欢的餐馆总是离我家'", 'raw_normalized': '我最喜欢的餐馆总是离我家', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'backend': 'transformers', 'task': 'cmnli', 'spli

,model_key,model_id,backend,task,split,samples,accuracy,macro_f1,invalid_rate,invalid_count,seconds,samples_per_second,label_id_style
0,gemma_e4b_it_4bit,mlx-community/gemma-4-e4b-it-4bit,mlx,afqmc,validation,200,0.185,0.148839,0.490,98,106.427360,1.879216,fixed_manual
1,gemma_e4b_it_4bit,mlx-community/gemma-4-e4b-it-4bit,mlx,tnews,validation,200,0.040,0.038410,0.410,82,112.532309,1.777267,fixed_manual_tnews
2,gemma_e4b_it_4bit,mlx-community/gemma-4-e4b-it-4bit,mlx,cmnli,validation,200,0.035,0.047343,0.900,180,110.915996,1.803166,fixed_manual
3,qwen3_4b_instruct_2507,Qwen/Qwen3-4B-Instruct-2507,transformers,afqmc,validation,200,0.670,0.658950,0.000,0,180.358688,1.108901,fixed_manual
4,qwen3_4b_instruct_2507,Qwen/Qwen3-4B-Instruct-2507,transformers,tnews,validation,200,0.405,0.359250,0.015,3,174.707507,1.144770,fixed_manual_tnews
5,qwen3_4b_instruct_2507,Qwen/Qwen3-4B-Instruct-2507,transformers,cmnli,validation,200,0.740,0.736331,0.000,0,145.230013,1.377126,fixed_manual
6,gemma_e2b_it,google/gemma-4-E2B-it,transformers,afqmc,validation,200,0.045,0.066241,0.890,178,109.464089,1.827083,fixed_manual
7,gemma_e2b_it,google/gemma-4-E2B-it,transformers,tnews,validation,200,0.015,0.001866,0.015,3,118.650511,1.685623,fixed_manual_tnews
8,gemma_e2b_it,google/gemma-4-E2B-it,transformers,cmnli,validation,200,0.025,0.030277,0.955,191,114.339693,1.749174,fixed_manual


In [ ]:
detail_df.head(20)


In [ ]:
invalid_df = detail_df[detail_df["pred_id"] == "__invalid__"].copy()
print("Invalid count:", len(invalid_df))
invalid_df.head(50)


In [ ]:
os.makedirs("results", exist_ok=True)

summary_path = "results/chinese_understanding_summary_v10.csv"
detail_path = "results/chinese_understanding_details_v10.csv"

summary_df.to_csv(summary_path, index=False)
detail_df.to_csv(detail_path, index=False)

print("Saved:")
print(summary_path)
print(detail_path)


## 9. Optional: run only one model

In [ ]:
# Example:
# one_key = "gemma_e4b_it_4bit"
# summaries, rows = evaluate_one_model(one_key, MODELS[one_key])
# pd.DataFrame(summaries)


## 10. Optional LoRA fine-tuning scaffold

In [ ]:
RUN_LORA_TRAINING = False

if RUN_LORA_TRAINING:
    from peft import LoraConfig
    from trl import SFTTrainer
    from transformers import TrainingArguments

    def format_sft_example(task, ex, spec):
        prompt = build_prompt(task, ex, spec)
        answer = label_id_to_name(ex["label"], spec)
        return prompt + answer

    def train_lora(model_id, task="afqmc", output_dir="adapters/lora_model", max_train_samples=1000):
        dataset = load_dataset("clue", task, split="train")
        dataset = dataset.select(range(min(max_train_samples, len(dataset))))
        spec = get_task_spec(task, dataset)

        dataset = dataset.map(lambda ex: {"text": format_sft_example(task, ex, spec)})

        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )

        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            task_type="CAUSAL_LM",
        )

        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            learning_rate=2e-4,
            num_train_epochs=1,
            logging_steps=20,
            save_steps=500,
            fp16=True,
            report_to="none",
        )

        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=dataset,
            dataset_text_field="text",
            peft_config=lora_config,
            args=training_args,
            max_seq_length=512,
        )

        trainer.train()
        trainer.save_model(output_dir)
        clear_model(model, tokenizer)

    # Example:
    # train_lora("Qwen/Qwen3-4B-Instruct-2507", task="afqmc", output_dir="adapters/qwen3_4b_afqmc")
